# Binary Variable Experiment

This notebook demonstrates the usage of the `conservative_pid` library to derive bounds for causal queries.
We consider a simple 2-binary variable setup (X, Y) where X causes Y.

In [1]:
%load_ext autoreload
%autoreload 2

from symbolic import Query
from solver import LPSolver


from canonical import VectorizedCanonicalBasis
import sys
import os

# Add parent directory to path to allow importing local modules
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
from symbolic import Variable, CounterfactualTerm, Event, P
from inference import ConservativePID

## 1. Setup Variables
We define two binary variables, X and Y, with domain {0, 1}.

In [2]:
X = Variable("X", (0, 1))
Y = Variable("Y", (0, 1))
variables = [X, Y]

## 2. Generate Synthetic Data
We generate data from a known Structural Causal Model (SCM):
- $P(X=1) = 0.5$
- $P(Y=1 | X=1) = 0.8$
- $P(Y=1 | X=0) = 0.3$

In [3]:
n_samples = 1000
np.random.seed(42)

x_data = np.random.binomial(1, 0.5, n_samples)

# Y depends on X
y_probs = np.where(x_data == 1, 0.8, 0.3)
y_data = np.random.binomial(1, y_probs, n_samples)

df = pd.DataFrame({"X": x_data, "Y": y_data})
df.head()

,X,Y
0,0,0
1,1,1
2,1,0
3,1,1
4,0,1


## 3. Estimate Relative Frequencies
We compute the empirical joint distribution $P(X, Y)$ from the data. 
This acts as the input `observational_data` for the inference engine.

In [4]:
# Compute counts
total = len(df)
observational_data = {}

for x_val in X.domain:
    for y_val in Y.domain:
        row_count = len(df[(df["X"] == x_val) & (df["Y"] == y_val)])
        observational_data[(x_val, y_val)] = row_count / total

print("Observational Data Distribution:", observational_data)

Observational Data Distribution: {(0, 0): 0.35, (0, 1): 0.153, (1, 0): 0.096, (1, 1): 0.401}


## 4. Inference with ConservativePID
We initialize the `ConservativePID` solver with the variables and the estimated distribution.

In [5]:
pid = ConservativePID(
    variables=variables,
    observational_data=observational_data,
)

### A. Observational Query
Query: $P(Y=1, X=1)$

In [6]:
# Target: (Y=1, X=1) (no intervention)
q_obs = P(Event({Y @ {}: 1}) & Event({X @ {}: 1}))
lb_obs, ub_obs = pid.infer(q_obs)

print(f"Observational Bounds P(Y=1,  X=1): [{lb_obs:.4f}, {ub_obs:.4f}]")
print(f"Relative Frequency: {observational_data[(1, 1)]:.4f}")

2026-01-27 20:26:44.386 | INFO     | inference:infer:83 - Found 2 valid causal orders compatible with the query.
2026-01-27 20:26:44.387 | INFO     | inference:infer:95 - Processing Order 1/2: ['Y', 'X']
2026-01-27 20:26:44.387 | INFO     | canonical:__init__:42 - Generated Basis with 8 worlds.
2026-01-27 20:26:44.441 | INFO     | inference:infer:95 - Processing Order 2/2: ['X', 'Y']
2026-01-27 20:26:44.442 | INFO     | canonical:__init__:42 - Generated Basis with 8 worlds.


Observational Bounds P(Y=1,  X=1): [0.4010, 0.4010]
Relative Frequency: 0.4010


Query: $P(Y=1 \mid X=1)$

In [7]:
# Target: Y=1 (no intervention)
# Evidence: X=1 (no intervention)
# q_obs = P(Event({Y @ {}: 1}), X == 1)
q_obs = P(X == 1, Y == 1)
lb_obs, ub_obs = pid.infer(q_obs)

print(f"Observational Bounds P(Y=1 | X=1): [{lb_obs:.4f}, {ub_obs:.4f}]")
print(f"Relative Frequency: {observational_data[(1, 1)]:.4f}")

2026-01-27 20:26:44.488 | INFO     | inference:infer:83 - Found 2 valid causal orders compatible with the query.
2026-01-27 20:26:44.488 | INFO     | inference:infer:95 - Processing Order 1/2: ['Y', 'X']
2026-01-27 20:26:44.488 | INFO     | canonical:__init__:42 - Generated Basis with 8 worlds.
2026-01-27 20:26:44.523 | INFO     | inference:infer:95 - Processing Order 2/2: ['X', 'Y']
2026-01-27 20:26:44.524 | INFO     | canonical:__init__:42 - Generated Basis with 8 worlds.


Observational Bounds P(Y=1 | X=1): [0.7238, 0.8068]
Relative Frequency: 0.4010


### B. Interventional Query
Query: $P(Y_{X=1} = 1)$
This represents the probability of Y=1 if we force X to be 1.

In [8]:
q_int = P(Y @ {X: 0} == 1)
lb_int, ub_int = pid.infer(q_int)

print(f"Interventional Bounds P(Y_{{X=1}} = 1): [{lb_int:.4f}, {ub_int:.4f}]")

2026-01-27 20:26:44.569 | INFO     | inference:infer:83 - Found 1 valid causal orders compatible with the query.
2026-01-27 20:26:44.570 | INFO     | inference:infer:95 - Processing Order 1/1: ['X', 'Y']
2026-01-27 20:26:44.570 | INFO     | canonical:__init__:42 - Generated Basis with 8 worlds.


Interventional Bounds P(Y_{X=1} = 1): [0.1530, 0.6500]


### C. Counterfactual Query
Query: $P(Y_{X=1} = 1 | X=0, Y=0)$
This asks: "Given that we observed X=0 and Y=0, what is the probability that Y would have been 1 if X had been 1?"

In [9]:
target = Event({Y @ {X: 1}: 1})
evidence = Event({X @ {}: 0, Y @ {}: 0})
q_cf = P(target, evidence)

lb_cf, ub_cf = pid.infer(q_cf)

print(f"Counterfactual Bounds P(Y_{{X=1}} = 1 | X=0, Y=0): [{lb_cf:.4f}, {ub_cf:.4f}]")

2026-01-27 20:26:44.617 | INFO     | inference:infer:83 - Found 1 valid causal orders compatible with the query.
2026-01-27 20:26:44.618 | INFO     | inference:infer:95 - Processing Order 1/1: ['X', 'Y']
2026-01-27 20:26:44.618 | INFO     | canonical:__init__:42 - Generated Basis with 8 worlds.


Counterfactual Bounds P(Y_{X=1} = 1 | X=0, Y=0): [0.0000, 1.0000]
